# 動画区間の切り出し

指定した動画から、指定した時間区間だけを新しい動画ファイルとして書き出します。候補動画の目視確認用クリップ作成を想定しています。

- 入力動画は変更・移動・削除しません。
- このノートブックは **OpenCVで再エンコードして出力** します。元ファイルと完全に同一の画質・音声を保持する方式ではありません。
- 上から順にセルを実行してください。


## 1. ライブラリを読み込む


In [ ]:
from pathlib import Path
import cv2


## 2. 切り出す動画と時間を指定する

- 時間は `時:分:秒`（例：`00:18:50`）または `分:秒`（例：`18:50`）で指定できます。
- `OUTPUT_DIR` は存在しなければ作成されます。


In [ ]:
# --- ここを変更 ---
VIDEO_PATH = Path(r"C:\path\to\input_video.mp4")
START_TIME = "00:18:50"
END_TIME = "00:19:20"
OUTPUT_DIR = Path(r"output_clips")

# 出力コーデック。WindowsのMP4出力では通常このままで使えます。
OUTPUT_CODEC = "mp4v"
# -----------------


## 3. 時間表記を秒へ変換する


In [ ]:
def time_to_seconds(time_text: str) -> int:
    """HH:MM:SS、MM:SS、または秒数を整数秒へ変換する。"""
    parts = time_text.strip().split(":")
    try:
        values = [int(part) for part in parts]
    except ValueError as exc:
        raise ValueError("時刻は 00:18:50 または 18:50 の形式で入力してください。") from exc

    if len(values) == 3:
        hours, minutes, seconds = values
    elif len(values) == 2:
        hours, minutes, seconds = 0, values[0], values[1]
    elif len(values) == 1:
        hours, minutes, seconds = 0, 0, values[0]
    else:
        raise ValueError("時刻は HH:MM:SS、MM:SS、または秒数で入力してください。")

    if minutes >= 60 or seconds >= 60 or min(values) < 0:
        raise ValueError("分・秒は0〜59、時刻は0以上で入力してください。")
    return hours * 3600 + minutes * 60 + seconds


def seconds_to_time(seconds: float) -> str:
    seconds = int(round(seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"

start_seconds = time_to_seconds(START_TIME)
end_seconds = time_to_seconds(END_TIME)
if start_seconds >= end_seconds:
    raise ValueError("開始時刻は終了時刻より前にしてください。")

print(f"切り出し範囲: {seconds_to_time(start_seconds)} 〜 {seconds_to_time(end_seconds)}")
print(f"予定時間: {end_seconds - start_seconds} 秒")


## 4. 入力動画の情報を確認する

ここで動画を開けること、時間範囲が動画の長さに収まっていることを確認します。


In [ ]:
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"動画が見つかりません: {VIDEO_PATH}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"動画を開けませんでした: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

if fps <= 0:
    raise RuntimeError("FPSを取得できませんでした。別の形式へ変換してから試してください。")

total_seconds = total_frames / fps
print(f"ファイル名: {VIDEO_PATH.name}")
print(f"解像度: {width} × {height}")
print(f"FPS: {fps:.3f}")
print(f"動画の長さ: {seconds_to_time(total_seconds)} ({total_seconds:.1f} 秒)")

if end_seconds > total_seconds:
    raise ValueError("終了時刻が動画の長さを超えています。設定を確認してください。")
print("動画と時間範囲の確認完了")


## 5. 切り出し後の出力先を確認する


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_name = (
    f"{VIDEO_PATH.stem}_{seconds_to_time(start_seconds).replace(':', '-')}_"
    f"{seconds_to_time(end_seconds).replace(':', '-')}.mp4"
)
output_path = OUTPUT_DIR / output_name
print(f"出力先: {output_path.resolve()}")
if output_path.exists():
    print("注意: 同名ファイルがあるため、実行すると上書きされます。")


## 6. 切り出しを実行する

入力動画は読み取るだけです。新しいMP4ファイルを出力します。


In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
start_frame = int(round(start_seconds * fps))
end_frame = int(round(end_seconds * fps))
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

fourcc = cv2.VideoWriter_fourcc(*OUTPUT_CODEC)
writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError("出力ファイルを作成できませんでした。出力先とコーデックを確認してください。")

written_frames = 0
try:
    for _ in range(end_frame - start_frame):
        ok, frame = cap.read()
        if not ok:
            print("注意: 動画の終端に達したため、予定より早く停止しました。")
            break
        writer.write(frame)
        written_frames += 1
finally:
    cap.release()
    writer.release()

print("切り出し完了")
print(f"出力ファイル: {output_path.resolve()}")
print(f"書き出しフレーム数: {written_frames}")
print(f"出力時間（目安）: {written_frames / fps:.1f} 秒")


## 注意点

- 音声トラックは出力されません。
- フレーム単位の厳密な編集・音声の保持・無劣化切り出しが必要な場合は、FFmpegなど別の方法が必要です。
- 作成したクリップを共有する際は、撮影施設・個体・利用許諾に関するルールを確認してください。
